In [1]:
import pyspark, os, sys, pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
STREAM_HOST = os.environ.get("SOCKET_HOST", "localhost")
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])
spark = SparkSession.builder.appName("DiscretizedStreamPrep").getOrCreate()
sc = spark.sparkContext
sc.setLogLevel("ERROR")
print("Spark version:", spark.version)
spark

Python /usr/local/bin/python
Java /usr/lib/jvm/java-17-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 20:54:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 20:54:13 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version: 4.1.2


# DStreams (Discretized Streams) -- Historical API and Migration to Structured Streaming

Spark originally provided two streaming APIs:

| API | Introduced | Status in Spark 4+ |
|-----|-----------|--------------------|
| **DStream** (Discretized Stream) | Spark 0.7 | **Downplayed** in Spark 4.0 |
| **Structured Streaming** | Spark 2.0 | Current standard |

Understanding DStreams is still valuable because:
- Much existing production code uses the DStream API.
- DStream concepts (micro-batches, batch interval, lineage, stateful operations) directly map to Structured Streaming concepts.
- Many interview questions and documentation references still mention DStreams.


# DStream Core Concepts

A **DStream** (Discretized Stream) is a sequence of RDDs, one per *batch interval*.

```
time: ────────────────────────────────────────────────────►
      [  RDD_t1  ][  RDD_t2  ][  RDD_t3  ][  RDD_t4  ]...
       <─ 10 s ─>  <─ 10 s ─>  <─ 10 s ─>  <─ 10 s ─>
            batch interval (set at StreamingContext creation)
```

Key components:
- **`StreamingContext(sc, batch_interval)`** -- the entry point.
- **Input DStream** -- created from a source (`socketTextStream`, `textFileStream`, Kafka, etc.).
- **Transformed DStream** -- result of `map`, `filter`, `flatMap`, `reduceByKey`, etc.
- **Output operation** -- `print()`, `saveAsTextFiles()`, `foreachRDD()`.

`StreamingContext.start()` kicks off the pipeline; `awaitTermination()` blocks until stopped.

# Example 1: DStream Basics -- Socket Source

The simplest DStream reads lines from a TCP socket and prints them each batch interval.


In [2]:
from pyspark.streaming import StreamingContext
ssc = StreamingContext(sc, 10)          # 10-second batch interval
lines = ssc.socketTextStream(STREAM_HOST, 7777)
lines.pprint()                          # print first 10 elements each batch

# Transformations
lines.map(lambda l: l.upper()).pprint()
lines.filter(lambda l: len(l) > 10).pprint()

# Window: reduce over last 30 s, evaluate every 10 s
lines.reduceByWindow(reduceFunc=lambda a, b: a + ", " + b,
                     invReduceFunc=None,
                     windowDuration=30, slideDuration=10).pprint()

ssc.start()
ssc.awaitTermination(30)
ssc.stop(stopSparkContext=False)
# ──────────────────────────────────────────────────────────────────────────
print("DStream reference code shown above (not executed -- API removed in Spark 4).")
print("The Structured Streaming equivalent follows.")

/usr/local/lib/python3.12/site-packages/pyspark/streaming/context.py:72: FutureWarning: DStream is deprecated as of Spark 3.4.0. Migrate to Structured Streaming.
  warnings.warn(
                                                                                

-------------------------------------------
Time: 2026-06-26 20:54:20
-------------------------------------------
hi there
streaming from terminal

-------------------------------------------
Time: 2026-06-26 20:54:20
-------------------------------------------
HI THERE
STREAMING FROM TERMINAL

-------------------------------------------
Time: 2026-06-26 20:54:20
-------------------------------------------
streaming from terminal

-------------------------------------------
Time: 2026-06-26 20:54:20
-------------------------------------------
hi there, streaming from terminal



-------------------------------------------
Time: 2026-06-26 20:54:30
-------------------------------------------
for a while
still here
but not sure when

-------------------------------------------
Time: 2026-06-26 20:54:30
-------------------------------------------
FOR A WHILE
STILL HERE
BUT NOT SURE WHEN

-------------------------------------------
Time: 2026-06-26 20:54:30
-------------------------------------------
for a while
but not sure when

-------------------------------------------
Time: 2026-06-26 20:54:30
-------------------------------------------
hi there, streaming from terminal, for a while, still here, but not sure when



-------------------------------------------
Time: 2026-06-26 20:54:40
-------------------------------------------
this will be cut
at some point
after 30 seconds

-------------------------------------------
Time: 2026-06-26 20:54:40
-------------------------------------------
THIS WILL BE CUT
AT SOME POINT
AFTER 30 SECONDS

-------------------------------------------
Time: 2026-06-26 20:54:40
-------------------------------------------
this will be cut
at some point
after 30 seconds

-------------------------------------------
Time: 2026-06-26 20:54:40
-------------------------------------------
hi there, streaming from terminal, for a while, still here, but not sure when, this will be cut, at some point, after 30 seconds



26/06/26 20:54:44 ERROR ReceiverTracker: Deregistered receiver for stream 0: Stopped by driver


DStream reference code shown above (not executed -- API removed in Spark 4).
The Structured Streaming equivalent follows.


[Stage 0:===========================================================(1 + 0) / 1]

# Example 2: DStream File Stream -- ISS Location Tracking

The Java course includes a data generator that calls the Open Notify API to get the real-time position of the International Space Station (ISS) and writes batches of JSON lines to files.

The `DStreamTransform` example processes these files with:
- `textFileStream(dir)` -- watches a directory for new files.
- `map` -- parse JSON to (latitude, longitude) pairs.
- `reduce` -- find the closest position to a reference point.
- `reduceByWindow` -- sliding minimum over a 300-second window.

**Java source:** `DStreamTransform.java`, `InternationalSpaceStationLocationStream.java`

We have sample ISS data in `data/iss_src/`.  Below we process it with **Structured Streaming**.

In [3]:
# Inspect the ISS sample data
import json as pyjson

with open("data/iss_src/1645910326846.txt") as f:
    lines = f.readlines()
print(f"Sample file has {len(lines)} records.")
for l in lines[:3]:
    print(pyjson.dumps(pyjson.loads(l), indent=2))

Sample file has 10 records.
{
  "timestamp": 1645910316,
  "iss_position": {
    "longitude": "-122.3257",
    "latitude": "-13.7918"
  },
  "message": "success"
}
{
  "timestamp": 1645910318,
  "iss_position": {
    "longitude": "-122.2681",
    "latitude": "-13.8665"
  },
  "message": "success"
}
{
  "timestamp": 1645910319,
  "iss_position": {
    "longitude": "-122.2297",
    "latitude": "-13.9162"
  },
  "message": "success"
}


In [4]:
# ── DStream textFileStream version (reference only -- Spark 4+) ───────────
from pyspark.streaming import StreamingContext
ssc = StreamingContext(sc, 10)          # 10-second batch interval

raw = ssc.textFileStream("data/iss_src")   # new files -> new RDD per batch

raw.pprint()

def parse(line):
    d = pyjson.loads(line)
    pos = d["iss_position"]
    return (float(pos["latitude"]), float(pos["longitude"]))

latlon = raw.map(parse)
latlon.pprint()

# Closest position to Athens, GA over the last 300 seconds
TCB = (33.9525, -83.3780)
dist = lambda p: ((p[0]-TCB[0])**2 + (p[1]-TCB[1])**2)**0.5
closest = latlon.map(lambda p: (dist(p), p)).reduceByWindow(
    lambda a, b: a if a[0] < b[0] else b,
    None,
    windowDuration=300, slideDuration=10)
closest.pprint()

ssc.start()
ssc.awaitTermination(20)
ssc.stop(stopSparkContext=False)
# ──────────────────────────────────────────────────────────────────────────
print("DStream textFileStream reference shown above.")

-------------------------------------------
Time: 2026-06-26 20:55:00
-------------------------------------------

-------------------------------------------
Time: 2026-06-26 20:55:00
-------------------------------------------

-------------------------------------------
Time: 2026-06-26 20:55:00
-------------------------------------------



-------------------------------------------
Time: 2026-06-26 20:55:10
-------------------------------------------
{"timestamp": 1645910316, "iss_position": {"longitude": "-122.3257", "latitude": "-13.7918"}, "message": "success"}
{"timestamp": 1645910318, "iss_position": {"longitude": "-122.2681", "latitude": "-13.8665"}, "message": "success"}
{"timestamp": 1645910319, "iss_position": {"longitude": "-122.2297", "latitude": "-13.9162"}, "message": "success"}
{"timestamp": 1645910320, "iss_position": {"longitude": "-122.1913", "latitude": "-13.9660"}, "message": "success"}
{"timestamp": 1645910321, "iss_position": {"longitude": "-122.1528", "latitude": "-14.0158"}, "message": "success"}
{"timestamp": 1645910322, "iss_position": {"longitude": "-122.1143", "latitude": "-14.0656"}, "message": "success"}
{"timestamp": 1645910323, "iss_position": {"longitude": "-122.0758", "latitude": "-14.1153"}, "message": "success"}
{"timestamp": 1645910324, "iss_position": {"longitude": "-122.0180", "lati

# Example 3: DStream Stateful Operations -- reduceByKeyAndWindow

`reduceByKeyAndWindow` is the DStream equivalent of a grouped sliding aggregation.  
It maintains state across batch windows without re-scanning all historical data.

```
DStream version:
words.flatMap(split)                          # one word per element
     .map(lambda w: (w, 1))                   # (word, 1) pairs
     .reduceByKeyAndWindow(add, 20s)          # sum over last 20 seconds

Structured Streaming equivalent:
words.select(explode(split(col("value"))))
     .withWatermark("ts", "20 seconds")
     .groupBy(window(col("ts"), "20 seconds", "10 seconds"), col("word"))
     .count()
```

In [5]:
from pyspark.streaming import StreamingContext
from operator import add

ssc = StreamingContext(sc, 10)
lines = ssc.socketTextStream(STREAM_HOST, 7777)

counts = (lines
    .flatMap(lambda line: line.split())
    .map(lambda word: (word, 1))
    .reduceByKeyAndWindow(add, None, windowDuration=20, slideDuration=10))

counts.pprint()
ssc.start()
ssc.awaitTermination(30)
ssc.stop(stopSparkContext=False)
print("DStream reduceByKeyAndWindow reference shown above.")

-------------------------------------------
Time: 2026-06-26 20:55:40
-------------------------------------------
('dog', 2)
('cat', 2)
('still', 1)
('there', 1)



-------------------------------------------
Time: 2026-06-26 20:55:50
-------------------------------------------
('mouse', 1)
('dog', 3)
('cat', 4)
('pig', 1)
('still', 1)
('there', 1)
('zebra', 1)



-------------------------------------------
Time: 2026-06-26 20:56:00
-------------------------------------------
('mouse', 1)
('dog', 3)
('cat', 4)
('pig', 1)
('zebra', 3)



26/06/26 20:56:03 ERROR ReceiverTracker: Deregistered receiver for stream 0: Stopped by driver


DStream reduceByKeyAndWindow reference shown above.


# Shutdown Spark when done

In [6]:
spark.stop()